# Неделя 3 — Baseline-модели

Карточка недели: `docs/week3_baseline.md` · Полное описание: тетрадь, раздел 7

**Конвенция ячеек:**
- ячейка без пометки — готовый код, просто запустите
- `# TODO:` — здесь пишете вы

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append('..')
from src.validation import (TEST_SNAPSHOTS, TRAIN_SNAPSHOTS, VALID_SNAPSHOTS,
                            evaluate, split_by_time)

dataset = pd.read_parquet('../data/processed/dataset.parquet')
print(dataset.shape)

(257631, 22)


## 1. Разбиение по времени с зазором

Схема — тетрада 7.2. Снапшоты 2025-04, 2025-07, 2025-10 выброшены намеренно:
иначе окно прогноза обучения пересекается с окном наблюдения теста.

Тест используется ровно один раз, в самом конце.

In [2]:
train, valid, test = split_by_time(dataset)
print(len(train), len(valid), len(test))
print('доля оттока:', train['target'].mean().round(4), test['target'].mean().round(4))

66253 35788 40265
доля оттока: 0.0762 0.1379


## 2. Baseline 0 — правило

Ничего не обучаем: просто ранжируем по одному признаку.
Минус обязателен — чем сильнее упал платёж, тем выше риск.

In [3]:
evaluate(test['target'], -test['revenue_last_to_mean'], 'Правило: падение платежа')

--- Правило: падение платежа ---
ROC-AUC: 0.8495
PR-AUC: 0.6476
Precision@10%: 0.7501
Lift@10%: 5.44x
Доля оттока: 0.1379


{'label': 'Правило: падение платежа',
 'roc_auc': 0.8494510949024486,
 'pr_auc': 0.6475775043196981,
 'precision_at_10': 0.7501241927471436,
 'lift_at_10': np.float64(5.440156812093783),
 'churn_rate': 0.13788650192474855}

## 3. Baseline 1 — логистическая регрессия

Обязательно через Pipeline + ColumnTransformer: импьютер и скейлер для числовых,
OneHotEncoder для категориальных. Без масштабирования результат будет мусорным.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Определяем колонки
drop_cols = ['target', 'client_id', 'snapshot_date']
cat_cols = ['segment', 'product', 'region']
num_cols = [c for c in test.columns if c not in drop_cols + cat_cols]

print(f"Числовые признаки ({len(num_cols)}): {num_cols}")
print(f"Категориальные признаки ({len(cat_cols)}): {cat_cols}")

# Пайплайн
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Подготовка данных
X_train = train.drop(columns=drop_cols)
y_train = train['target']
X_test = test.drop(columns=drop_cols)
y_test = test['target']

# Обучение и оценка
logreg_pipeline.fit(X_train, y_train)
y_pred_proba = logreg_pipeline.predict_proba(X_test)[:, 1]

evaluate(y_test, y_pred_proba, 'Логистическая регрессия')

Числовые признаки (16): ['revenue_mean', 'revenue_min', 'revenue_max', 'revenue_std', 'revenue_count', 'n_sim_last', 'n_sim_mean', 'debt_mean', 'debt_last', 'revenue_last', 'revenue_last_to_mean', 'revenue_trend', 'months_declining', 'revenue_trend_norm', 'n_sim_trend', 'debt_months']
Категориальные признаки (3): ['segment', 'product', 'region']
--- Логистическая регрессия ---
ROC-AUC: 0.8799
PR-AUC: 0.7059
Precision@10%: 0.7655
Lift@10%: 5.55x
Доля оттока: 0.1379


{'label': 'Логистическая регрессия',
 'roc_auc': 0.879872291198698,
 'pr_auc': 0.705869480659583,
 'precision_at_10': 0.7655240933929458,
 'lift_at_10': np.float64(5.551842150620211),
 'churn_rate': 0.13788650192474855}

## 4. Сравнение baseline'ов

TODO: таблица метрик — правило vs логрегрессия.